# Direct loss estimation (DLE)

`DirectLossEstimator` learns squared loss from labeled reference observations and compares estimated current loss with a held-out reference baseline. For regression, the mean loss is **MSE**. For binary classification with labels 0/1 and `y_pred = P(y=1)`, it is the **Brier score**.

The held-out reference is split into contiguous chunks. `StatisticalInterval` computes a limit from their mean estimated losses. `degradation` is true when the current estimated mean exceeds both this `reference_limit` and the relative `degradation_margin`. Choose `chunk_size` close to the usual current batch size (default 50) and keep enough reference rows for several chunks. Available `interval_method` values include `"stddev"`, `"mad"`, and `"iqr"`. The alert concerns estimated loss; compare with realized performance when current targets arrive.

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
from sklearn.ensemble import RandomForestRegressor

from tinyshift.performance import DirectLossAnalyzer, DirectLossEstimator

rng = np.random.default_rng(42)

## 1. DLE for regression vectors

The monitored model predicts a numeric target. Its squared error grows with `risk`, and the current batch contains more high-risk observations. The DLE fits its learner on the first half of reference rows and uses the last half as a baseline. Six reference chunks of 100 rows are comparable to the 120-row current batch.

In [2]:
reference_risk = rng.uniform(0, 1, 1200)
current_risk = rng.beta(8, 2, 120)
X_reference = reference_risk.reshape(-1, 1)
X_current = current_risk.reshape(-1, 1)
predictions_reference = 10 + 0.5 * reference_risk
predictions_current = 10 + 0.5 * current_risk
y_reference = predictions_reference + 0.3 + 1.5 * reference_risk + rng.normal(0, 0.04, 1200)

regression_dle = DirectLossEstimator(
    learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=5, random_state=42),
    fraction=0.5,
    chunk_size=100,
).fit(X_reference, y_reference, predictions_reference)
regression_result = regression_dle.predict(
    X_current, predictions_current, degradation_margin=0.10
)
regression_result

DirectLossResult(reference_estimated=1.3060004126035896, reference_realized=1.3189693801616285, reference_size=600, current_estimated=2.149723257248304, estimated_delta=0.8437228446447143, relative_delta=0.6460356646922512, degradation_margin=0.1, reference_limit=1.5928166196580378, degradation=True, current_size=120)

`regression_result` reports `reference_estimated`, `current_estimated`, `relative_delta`, `reference_limit`, and the resulting `degradation` flag. The reference limit is learned from `regression_dle.reference_chunk_losses_`, which contains the six chunk means. To inspect the numeric current estimate or per-row estimates, use `estimate` or `estimate_loss`:

In [3]:
regression_dle.estimate(X_current, predictions_current), regression_dle.estimate_loss(X_current, predictions_current)[:5]

(2.149723257248304,
 array([0.93465288, 2.792048  , 2.40429083, 1.87488822, 2.44576371]))

The margin can suppress an alert even when the current loss exceeds the reference limit. Raising it to 80% in this example requires a larger operational change. A `False` decision does not establish that performance is unchanged.

In [4]:
regression_dle.predict(
    X_current, predictions_current, degradation_margin=0.80
)

DirectLossResult(reference_estimated=1.3060004126035896, reference_realized=1.3189693801616285, reference_size=600, current_estimated=2.149723257248304, estimated_delta=0.8437228446447143, relative_delta=0.6460356646922512, degradation_margin=0.8, reference_limit=1.5928166196580378, degradation=False, current_size=120)

## 2. DLE for binary probability vectors

Use `P(y=1)` as the prediction vector and numeric labels 0/1. The current probabilities are closer to 0.5, where a calibrated classifier generally has higher expected Brier loss. Six reference chunks of 200 rows match the 200-row current batch. The result uses the same fields as in regression.

In [5]:
binary_rng = np.random.default_rng(18)
p_reference = binary_rng.beta(1.5, 1.5, 2400)
p_current = binary_rng.beta(10, 10, 200)
y_binary_reference = binary_rng.binomial(1, p_reference)
X_binary_reference = np.zeros((len(p_reference), 1))
X_binary_current = np.zeros((len(p_current), 1))

brier_dle = DirectLossEstimator(
    learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=10, random_state=42),
    fraction=0.5,
    chunk_size=200,
).fit(X_binary_reference, y_binary_reference, p_reference)
brier_dle.predict(X_binary_current, p_current, degradation_margin=0.20)

DirectLossResult(reference_estimated=0.18631079598357597, reference_realized=0.1880252997308689, reference_size=1200, current_estimated=0.23482235307912178, estimated_delta=0.048511557095545804, relative_delta=0.2603797425664065, degradation_margin=0.2, reference_limit=0.20286445382292786, degradation=True, current_size=200)

## 3. Analyzer for regression panels

`DirectLossAnalyzer` clones one DLE per ID and collects results in a DataFrame. Each ID has its own reference chunks, interval limit, and current comparison. Preserve the intended row order; for a time series, order rows chronologically within each ID before fitting.

In [6]:
def regression_batch(store, risk, labeled):
    prediction = 10 + 0.5 * risk
    frame = pd.DataFrame({"unique_id": store, "risk": risk, "y_pred": prediction})
    if labeled:
        error = 0.3 + 1.5 * risk if store == "store_A" else 0.6 + 0.5 * risk
        frame["y"] = prediction + error + rng.normal(0, 0.04, len(risk))
    return frame


regression_reference = pd.concat([
    regression_batch("store_A", rng.uniform(0, 1, 1200), labeled=True),
    regression_batch("store_B", rng.uniform(0, 1, 1200), labeled=True),
], ignore_index=True)
regression_current = pd.concat([
    regression_batch("store_A", rng.beta(8, 2, 120), labeled=False),
    regression_batch("store_B", rng.beta(2, 8, 120), labeled=False),
], ignore_index=True)

regression_analyzer = DirectLossAnalyzer(
    estimator=DirectLossEstimator(
        learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=5, random_state=42),
        chunk_size=100,
    ),
    fraction=0.5,
).fit(regression_reference, feature_cols=["risk"])
regression_analyzer.predict(regression_current, degradation_margin=0.10)

,unique_id,reference_estimated,reference_realized,reference_size,current_estimated,estimated_delta,relative_delta,degradation_margin,reference_limit,degradation,current_size
0,store_A,1.266410,1.260721,600,2.313288,1.046878,0.826650,0.1,1.680905,True,120
1,store_B,0.732063,0.732644,600,0.485486,-0.246577,-0.336825,0.1,0.815951,False,120


The result contains one row per ID. Compare `relative_delta` with `degradation_margin` and `current_estimated` with that ID's `reference_limit`; `degradation` is true only when both comparisons hold. The fitted limits are also available as `regression_analyzer.estimators_[id].reference_limit_`.

## 4. Analyzer for binary probability panels

The same analyzer accepts one `y_pred` probability column and 0/1 reference labels. Here model A becomes less confident and model B more confident in the current batch. The output loss is Brier score, because it averages `(y - P(y=1))²`. Each ID uses six 200-row reference chunks against a 200-row current batch.

In [7]:
panel_rng = np.random.default_rng(29)


def binary_batch(model_id, probabilities, labeled):
    frame = pd.DataFrame({
        "unique_id": model_id,
        "segment": np.zeros(len(probabilities)),
        "y_pred": probabilities,
    })
    if labeled:
        frame["y"] = panel_rng.binomial(1, probabilities)
    return frame


binary_reference = pd.concat([
    binary_batch("model_A", panel_rng.beta(1.5, 1.5, 2400), labeled=True),
    binary_batch("model_B", panel_rng.beta(1.5, 1.5, 2400), labeled=True),
], ignore_index=True)
binary_current = pd.concat([
    binary_batch("model_A", panel_rng.beta(10, 10, 200), labeled=False),
    binary_batch("model_B", panel_rng.beta(0.5, 0.5, 200), labeled=False),
], ignore_index=True)

brier_analyzer = DirectLossAnalyzer(
    estimator=DirectLossEstimator(
        learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=10, random_state=42),
        chunk_size=200,
    ),
    fraction=0.5,
).fit(binary_reference, feature_cols=["segment"])
brier_analyzer.predict(binary_current, degradation_margin=0.10)

,unique_id,reference_estimated,reference_realized,reference_size,current_estimated,estimated_delta,relative_delta,degradation_margin,reference_limit,degradation,current_size
0,model_A,0.186739,0.192363,1200,0.238497,0.051759,0.277171,0.1,0.205552,True,200
1,model_B,0.189596,0.189118,1200,0.119163,-0.070433,-0.371490,0.1,0.203090,False,200


Current targets are absent in these panels, so realized current performance cannot yet be checked. Estimated loss can become inaccurate if the relationship between inputs and loss changes; changed probability calibration is one example. Compare estimates with realized loss when labels arrive. A `False` alert does not prove that performance stayed constant.